# Bài 21 — chuỗi luyện Easy · LeetCode #111 — Minimum Depth of Binary Tree

**Mức độ:** Easy · **Chủ đề:** Tree · BFS · Queue · **Python:** 3  
**Đề gốc:** [https://leetcode.com/problems/minimum-depth-of-binary-tree/](https://leetcode.com/problems/minimum-depth-of-binary-tree/)

> Mục tiêu: hiểu cách tự suy luận từ ví dụ → viết code → trace trạng thái → chứng minh đúng → tự làm bài biến thể. Đọc Markdown, dự đoán rồi mới chạy cell code tiếp theo.

**Lưu ý:** đây là notebook dành cho người mới. Các cell TODO chỉ định nghĩa hàm, không được tự gọi khi chưa giải.

**Lưu ý học:** các ô TODO không được tự chạy; hãy tự làm, mở gợi ý từng mức rồi mới xem đáp án ở cuối.


**Lộ trình đầy đủ:** đề → nền tảng → baseline → điểm mấu chốt → code → trace → chứng minh → nộp LeetCode → Big-O → lỗi → biến thể → 6 bài tập → gợi ý → đáp án → checklist.

## 1. Đọc đề và xác định đúng đầu vào/đầu ra

Độ sâu tối thiểu là số node trên đường từ gốc đến **lá gần nhất**. Lá là node không có **cả hai** con; một nhánh trống của node chưa phải lá.

### Ví dụ quan trọng

`[3,9,20,None,None,15,7] → 2` (gốc→9); `[2,None,3,None,4] → 3`, không phải 1.

Hãy tự trả lời: **đầu ra là số, boolean, danh sách, hay node?** Với bài cấu trúc dữ liệu, phải chú ý **node/tham chiếu** khác **giá trị số**.

## 2. Thiết lập môi trường để chạy notebook

Trong Jupyter, hãy chạy cell tạo lớp dữ liệu và hàm hỗ trợ trước các cell sau. **Lưu ý:** LeetCode có sẵn một số lớp hoặc API; khi submit chỉ sao chép riêng cell `class Solution`/lớp thiết kế, không sao chép hàm giả lập dùng cho notebook.

In [ ]:
from __future__ import annotations
from typing import Optional
from collections import deque

class TreeNode:
    def __init__(self, val: int = 0, left=None, right=None):
        self.val, self.left, self.right = val, left, right
    def __repr__(self):
        return f"TreeNode({self.val})"

def tree(values):
    """Tạo cây từ danh sách level-order; None = ô trống."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); queue=deque([root]); pos=1
    while queue and pos < len(values):
        parent=queue.popleft()
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.left=TreeNode(v); queue.append(parent.left)
        if pos < len(values):
            v=values[pos]; pos+=1
            if v is not None:
                parent.right=TreeNode(v); queue.append(parent.right)
    return root

def show_tree(root):
    """Đọc cấu trúc cây bằng BFS, không in quá nhiều None cuối."""
    if root is None:
        return []
    q=deque([root]); result=[]
    while q:
        p=q.popleft()
        if p is None:
            result.append(None)
        else:
            result.append(p.val); q.append(p.left); q.append(p.right)
    while result and result[-1] is None:
        result.pop()
    return result

### 2.1. Từ một nút đến cây nhị phân (Binary Tree)

*TreeNode* là một **đối tượng** có 3 thuộc tính: `.val` giữ số, `.left` tham chiếu con trái, `.right` tham chiếu con phải.
Hai con có thể là `None` nghĩa là nút con không tồn tại. Nút không có cả hai con gọi là **lá** (*leaf*).
Một đường gốc → lá là chuỗi các nút liên tiếp nối bằng con trỏ; không được nhảy qua nút.

Ví dụ `root=TreeNode(5)` rồi gắn hai con 4 và 8: gốc ở **độ sâu 1**, hai con ở **độ sâu 2**. 
Đếm độ sâu theo **số nút** trên đường, không theo số cạnh. Cây rỗng có độ sâu 0.
Trước khi chạy, dự đoán giá trị của 5 phép đọc thuộc tính sau.

In [ ]:
simple_root = TreeNode(5)
simple_root.left = TreeNode(4)
simple_root.right = TreeNode(8)
print(simple_root.val, simple_root.left.val, simple_root.right.val)
print('con trái của 4:',simple_root.left.left)
print('gốc có là lá?',simple_root.left is None and simple_root.right is None)
print('nút 4 có là lá?', simple_root.left.left is None and simple_root.left.right is None)

### 2.2. Vì sao phải kiểm tra `None` trước khi truy cập `.val`?

`None` nghĩa là không có đối tượng. Viết `node.val` khi `node is None` sẽ gây `AttributeError`.
Vì thế thuật toán cây thường mở bằng `if node is None: ...` để làm **trường hợp dừng**.
`is None` kiểm tra chính xác giá trị đặc biệt `None`; `if node:` kiểm tra tính chân trị và đôi khi gây nhầm khi
đối tượng có định nghĩa quy tắc `bool` riêng.

Dấu `and` và `or` có **short-circuit**: Python dừng đánh giá phần sau khi đã biết kết quả. 
Ví dụ `node is not None and node.val > 0` an toàn: nếu node rỗng thì không thử đọc `.val`.
Còn `node.val > 0 and node is not None` không an toàn vì đổi thứ tự!

In [ ]:
missing_demo = None
present_demo = TreeNode(0)
for candidate in (missing_demo,present_demo,TreeNode(-3),TreeNode(7)):
    ok = candidate is not None and candidate.val > 0
    print('ứng viên:',candidate,'-> giá trị dương?',ok)
assert missing_demo is None

### 2.3. Tham chiếu và sự khác biệt giữa giá trị / vị trí nút

`a = root.left` không tạo ra một bản sao: `a` chỉ là tên khác tham chiếu cùng đối tượng.
Trong cây, **vị trí** một nút quan trọng hơn việc một số xuất hiện bao nhiêu lần; hai nút có cùng `.val`
vẫn là hai đối tượng khác nhau. `is` hỏi có phải cùng đối tượng; `==` hỏi hai giá trị bằng nhau.

Khi giải bài đường đi, ta cần đi theo `left/right` thực sự, không được lấy các số từ mảng mô tả cây rồi
cộng/so theo thứ tự phần tử. Dữ liệu `[1,2,None,3]` biểu diễn cây khác `[1,None,2,3]`.

In [ ]:
a_node = TreeNode(9)
b_node = TreeNode(9)
alias = a_node
print('hai giá trị:', a_node.val == b_node.val)
print('có cùng đối tượng:',a_node is b_node)
print('alias có cùng đối tượng:',alias is a_node)
print('hai cấu trúc khác:', show_tree(tree([1,2,None,3])),show_tree(tree([1,None,2,3])))

### 2.4. Danh sách level-order khác với cây trong bộ nhớ

LeetCode viết cây dưới dạng danh sách *level-order* (từng tầng, trái đến phải), chèn `null`
khi một nhánh không tồn tại. Ở Python notebook này dùng `None` thay chữ `null` trong đề.
Hàm `tree(values)` chỉ là công cụ kiểm thử: lần lượt lấy **nút cha** khỏi queue rồi gắn hai nút con.

Khi nộp LeetCode, tham số `root` **đã là TreeNode**, không phải `list`.
Vì vậy lời giải không cần `tree(...)`, cũng không được truy cập `root[0]`.
`show_tree` là hàm phụ giúp nhìn cấu trúc; không cần nộp nó.

In [ ]:
shape1=tree([1,2,3,None,4])
print('gốc',shape1,'trái',shape1.left,'phải',shape1.right)
print('con phải của 2',shape1.left.right)
print('đọc lại bằng level-order:',show_tree(shape1))
print('cây rỗng:',tree([]),show_tree(None))

### 2.5. Đệ quy từ đầu: trường hợp dừng, lời gọi con, kết quả trả về

**Recursion (đệ quy)** là hàm gọi lại chính nó nhưng trên đầu vào nhỏ hơn.
Hai điều kiện để không chạy vô hạn: (1) có **base case**; (2) mỗi lời gọi con tiến gần base case.
Với cây: `None` hoặc lá là trường hợp nhỏ nhất; `node.left` / `node.right` là cây con nhỏ hơn.

**Call stack** giữ các lời gọi đang chờ `return`. Nếu cây có chiều cao `h`, đệ quy DFS có thể cần `O(h)`
bộ nhớ ngăn xếp. Trong Python, cây quá lệch có thể chạm giới hạn đệ quy;
BFS dùng queue tránh vấn đề ấy, còn DFS lặp dùng stack tường minh.

In [ ]:
def countdown_foundation(n,indent=0):
    print('  '*indent+'vào',n)
    if n == 0:
        return 0
    child = countdown_foundation(n-1,indent+1)
    result = child + 1
    print('  '*indent+'quay lại',n,'kết quả',result)
    return result
assert countdown_foundation(3)==3

### 2.6. `return` khác `print`: LeetCode cần giá trị đúng

`print(x)` chỉ hiển thị x trên màn hình; lệnh không trả x từ hàm.
`return x` vừa kết thúc lời gọi hiện tại, vừa gửi x về nơi gọi.
Type hint như `Optional[TreeNode]` nói tham số có thể là `TreeNode` hoặc `None`;
nó giúp đọc code nhưng không thay phần kiểm tra thực tế.

Một hàm trả `bool` phải thực sự `return True/False`; hàm trả `int` phải trả con số đúng.
Đừng đặt mọi phép `print` vào hàm nộp LeetCode: chúng chỉ phục vụ quan sát trong notebook.

In [ ]:
def only_print_demo(x):
    print('bên trong:',x)
def really_return_demo(x):
    return x
print('kết quả chỉ print:', only_print_demo(42))
print('kết quả return:', really_return_demo(42))
assert only_print_demo(0) is None
assert really_return_demo(0) == 0

### 2.7. Tầng sâu, số nút và cách tạo cây phản ví dụ

Ký hiệu `n` là tổng nút thật, `h` là độ cao lớn nhất, `w` là số nút đông nhất trong một tầng.
`n`, `h`, `w` không phải cùng một đại lượng.

- Cây hình dây có `h=n`, mỗi tầng tối đa một nút.
- Cây rất đầy có `h` nhỏ (cỡ log₂ n), nhưng tầng cuối có thể chứa nhiều nút.
- Khi thuật toán cần **ngắn nhất đến lá**, một nhánh `None` **không phải** là một lá.
- Khi bài hỏi **tổng đường đi**, tổng của node ở giữa đường không đủ nếu chưa chạm lá.

Hãy dùng hai cây sau để dự đoán độ sâu tối đa và vị trí lá; đây là cách dựng phản ví dụ có chủ đích.

In [ ]:
chain_demo = tree([1,None,2,None,3,None,4])
fork_demo = tree([1,2,3,4,None,None,5])
for root_demo in (chain_demo,fork_demo):
    q_demo=deque([(root_demo,1)])
    leaf_info=[]
    while q_demo:
        node,depth=q_demo.popleft()
        if node.left is None and node.right is None:leaf_info.append((node.val,depth))
        if node.left:q_demo.append((node.left,depth+1))
        if node.right:q_demo.append((node.right,depth+1))
    print('cây:',show_tree(root_demo),'-> các lá và độ sâu:',leaf_info)

## 3. Python nền tảng — công cụ cần hiểu trước khi giải

`collections.deque`: hàng đợi FIFO; `append` thêm cuối, `popleft` lấy đầu. Duyệt BFS theo tầng: ai được lấy trước cũng là node có độ sâu không lớn hơn node sau.

**Bài thí nghiệm:** thử thay dữ liệu trong cell dưới; các lệnh `print()` dùng để thấy giá trị trung gian, không phải mã submit.

In [ ]:
q=deque([(10,1)]);q.append((20,2));q.append((30,2))
print('lấy trước:',q.popleft());print('còn lại:',list(q))
r=tree([2,None,3]);print('gốc là lá?',r.left is None and r.right is None)

### 3.1. Queue FIFO là gì? `append` và `popleft` khác gì?

Queue (hàng đợi) làm việc theo quy tắc **First In, First Out**: ai vào trước thì được xử lý trước.
`deque` từ module `collections` hỗ trợ `append` ở cuối và `popleft` ở đầu, mỗi thao tác O(1).
Nếu `queue=[A,B]`, thêm C thì thành `[A,B,C]`, lấy ra đầu là A, còn `[B,C]`.
Đó là chìa khóa của **BFS (Breadth-First Search)**: bắt đầu ở gốc rồi lần lượt thăm các nút theo tầng.

Vì sao không dùng stack `pop()` ở cuối? Stack sẽ xét nhánh mới nhất trước, đi sâu xuống một nhánh; đó là **DFS**,
không đảm bảo lá đầu tiên được thấy là lá nông nhất.

In [ ]:
from collections import deque
queue_tutorial=deque(['A'])
queue_tutorial.append('B')
queue_tutorial.append('C')
print('ban đầu:',list(queue_tutorial))
first=queue_tutorial.popleft()
print('lấy đầu:',first,'còn:',list(queue_tutorial))
assert first=='A' and list(queue_tutorial)==['B','C']

### Tự kiểm tra ý nghĩa từng dòng

- Đầu vào của mỗi biểu thức là gì?
- Dòng nào tạo cấu trúc mới và dòng nào thay đổi cấu trúc hiện có?
- `True`/`False` và `None` có nghĩa gì trong trường hợp này?
- Nếu dữ liệu rỗng hoặc chỉ có một phần tử, đoạn mã có còn chạy được không?

## 4. Cách nghĩ trực giác (baseline)

Dùng đệ quy: với `None` trả 0, nhưng nếu **chỉ có một nhánh**, phải đi nhánh còn lại; không thể `1+min(0, depth(right))`.

**Hãy dự đoán** kết quả của hàm baseline trên các ví dụ trước khi chạy.

In [ ]:
def baseline(root):
    if root is None:return 0
    if root.left is None:return 1+baseline(root.right)
    if root.right is None:return 1+baseline(root.left)
    return 1+min(baseline(root.left),baseline(root.right))

In [ ]:
for a in [[],[3,9,20,None,None,15,7],[2,None,3,None,4]]:
    print(a,'->',baseline(tree(a)))

### 4.1. Phản ví dụ quan trọng: nhánh rỗng không phải đường đến lá

Một sai lầm rất dễ xảy ra: định nghĩa `minDepth(node)=1+min(minDepth(left),minDepth(right))` cho mọi node.
Nếu cây là `1 → 2 → 3` chỉ có con phải, thì `minDepth(left)=0` ở gốc, nên công thức sai trả `1`.
Nhưng **không có con trái**, ta không thể đi từ gốc đến một lá qua một chỗ trống.
Đường hợp lệ duy nhất là `1→2→3`, gồm ba nút, nên đáp án phải là 3.

**Quy tắc:** chỉ khi *cả hai con tồn tại* mới so sánh `min` của hai độ sâu; nếu chỉ có một con thì đi con đó.
Trái lại, BFS không cần trường hợp công thức đặc biệt vì nó chỉ đưa node thật vào queue.

In [ ]:
chain_for_counterexample=tree([1,None,2,None,3])
def wrong_min_depth_demo(node):
    if node is None:return 0
    return 1+min(wrong_min_depth_demo(node.left),wrong_min_depth_demo(node.right))
print('công thức sai:',wrong_min_depth_demo(chain_for_counterexample))
print('công thức đúng:',baseline(chain_for_counterexample))
assert wrong_min_depth_demo(chain_for_counterexample)==1
assert baseline(chain_for_counterexample)==3

### 4.2. Baseline DFS đi hết hai nhánh; BFS có thể dừng sớm

Hàm baseline dùng đệ quy DFS (Depth-First Search): giải cây con trước rồi gộp kết quả.
Nó đúng nếu xử lý nút một con chính xác, và có cận O(n) thời gian, O(h) bộ nhớ call stack.
Tuy nhiên DFS không biết ngay lá bên trái ở độ sâu 2 là đáp án nhỏ nhất: nó có thể tính cả cây con phải rất sâu.

Ví dụ gốc có con trái là lá, con phải là một chuỗi dài: minDepth chắc chắn bằng 2,
BFS chỉ cần xem gốc rồi lá trái, còn DFS theo công thức thường tính toàn bộ cây con phải.
Đây là **tối ưu dừng sớm** chứ không phải thay đổi Big-O tệ nhất: trường hợp xấu BFS vẫn xét n nút.

In [ ]:
branching_tree=tree([1,2,3,None,None,4,None,5,None,6])
print('cây:',show_tree(branching_tree))
print('DFS baseline:',baseline(branching_tree))
print('số nút (có nhánh sâu):',len([v for v in show_tree(branching_tree) if v is not None]))
assert baseline(branching_tree)==2

## 5. Vì sao cần chuyển sang kỹ thuật chính?

BFS duyệt theo **thứ tự độ sâu tăng dần**. Lá đầu tiên lấy ra khỏi queue chính là lá nông nhất: có thể dừng sớm ngay khi gặp.

Một thuật toán tốt cần trả lời: **ta đang nhớ thông tin gì, cập nhật khi nào, và làm sao không bỏ sót trường hợp nào?**

## 6. Lời giải chính — hiểu từng bước rồi mới viết

Queue lưu `(node,depth)`. Bắt đầu `(root,1)`; lấy ra, nếu là lá thì trả `depth`; nếu không, đẩy **chỉ những node con thực sự tồn tại** với `depth+1`.

Dưới đây là **hàm giải độc lập**. Khi submit, ta đưa cùng logic vào đúng chữ ký LeetCode.

### 5.1. Vì sao BFS luôn lấy nút nông trước nút sâu?

Chứng minh bằng một thao tác queue nhỏ. Ban đầu queue giữ duy nhất gốc độ sâu 1.
Khi lấy một nút tầng d, ta **chỉ thêm con của nó** ở tầng d+1 vào cuối hàng đợi.
Do các nút đang chờ ở tầng d có mặt trước các con vừa thêm, tất cả tầng d được xử lý trước tầng d+1.
Lặp lại lập luận ấy, ta duyệt tầng 1, rồi 2, rồi 3, ...

Do đó khi lấy một **lá đầu tiên** ở tầng d, không có lá nào ở tầng nhỏ hơn còn chưa được xét;
nếu có, nó đã khiến thuật toán trả sớm hơn. Đó là lý do BFS giải minDepth đúng.

In [ ]:
root_layers=tree([8,3,10,1,6,None,14])
q_layers=deque([(root_layers,1)])
seen_layers=[]
while q_layers:
    node,depth=q_layers.popleft(); seen_layers.append((node.val,depth))
    if node.left:q_layers.append((node.left,depth+1))
    if node.right:q_layers.append((node.right,depth+1))
print('thứ tự BFS:',seen_layers)
assert [d for _,d in seen_layers]==sorted(d for _,d in seen_layers)

In [ ]:
def solve(root: Optional[TreeNode]) -> int:
    if root is None:return 0
    q=deque([(root,1)])
    while q:
        node,depth=q.popleft()
        if node.left is None and node.right is None:
            return depth
        if node.left is not None:q.append((node.left,depth+1))
        if node.right is not None:q.append((node.right,depth+1))
    return 0

### 6.1. Mổ xẻ từng biến trong hàm `solve` vừa viết

`q` là queue chờ xử lý, mỗi phần tử là tuple `(node, depth)`.
`while q:` nghĩa là lặp khi queue vẫn còn phần tử.
`node, depth = q.popleft()` là *tuple unpacking*: tách hai thành phần thành hai biến.
Tại mọi bước, `depth` là độ sâu **tính theo số nút** từ gốc đến node.

Điều kiện lá phải kiểm tra **cả hai con rỗng**. Nếu là lá, trả độ sâu ngay.
Nếu chưa phải lá, chỉ `append` các con **khác None** với `depth+1`.
Hàng đợi không chứa các nút `None`, không chứa giá trị null giả lập, và không tự sửa cây gốc.

In [ ]:
node_tuple=(TreeNode(12),4)
node_unpack, depth_unpack=node_tuple
print('node=',node_unpack,'depth=',depth_unpack)
print('có phải lá?',node_unpack.left is None and node_unpack.right is None)
print('tăng depth cho con giả định:',depth_unpack+1)
assert depth_unpack==4

### 6.2. Một nút đơn và cây rỗng: hai base case khác nhau

Cây rỗng có `root is None`: không có đường gốc đến lá, nên theo quy ước đề bài trả 0.
Cây có đúng một nút: gốc đồng thời là lá, độ sâu nhỏ nhất bằng 1.
Ở cây chỉ có hai nút gốc→con phải, lá ở độ sâu 2, dù phía trái rỗng.

Trực giác “chỗ trống tính là tầng 0” là đúng khi tính chiều cao cây rỗng,
nhưng tuyệt đối không dùng chỗ trống làm **đích** của bài minDepth.
Hãy tự nói câu này trước khi xem output để không mắc lỗi toán học ở phần biến thể.

In [ ]:
for values in ([],[7],[7,None,9],[7,8,9]):
    print('input:',values,'-> minDepth=',solve(tree(values)))
assert solve(None)==0
assert solve(TreeNode(7))==1
assert solve(tree([7,None,9]))==2

## 7. Trace bằng tay — quan sát trạng thái, không chỉ đáp án

| Lấy từ queue | depth | Có phải lá? | Hành động |
|---|---:|---|---|
| `3` | 1 | không | thêm `(9,2),(20,2)` |
| `9` | 2 | **có** | **trả 2 ngay** |
| `20` | 2 | không cần duyệt | kết quả đã biết |

Đọc từ trái sang phải: mỗi dòng ghi **trạng thái trước khi quyết định**, hành động và phần thay đổi **sau khi quyết định**.

### 7.1. Trace bằng bảng đầy đủ, giữ cả queue trước/sau

Với `[1,2,3,4,None,None,5]`, giả sử queue lưu các cặp `(giá trị nút,độ sâu)`.

| Bước | Queue trước | Nút lấy ra | Lá? | Queue sau |
|---:|---|---|---|---|
| 1 | `[(1,1)]` | `(1,1)` | Không | `[(2,2),(3,2)]` |
| 2 | `[(2,2),(3,2)]` | `(2,2)` | Không | `[(3,2),(4,3)]` |
| 3 | `[(3,2),(4,3)]` | `(3,2)` | Không | `[(4,3),(5,3)]` |
| 4 | `[(4,3),(5,3)]` | `(4,3)` | **Có** | return `3` |

Điểm mấu chốt: queue có thể còn phần tử khi trả lời. Vì chỉ cần lá nông nhất,
không có lý do đi hết các lá sâu hơn. Hãy so các dòng bảng với output thực tế.

In [ ]:
def bfs_queue_snapshot(root):
    q=deque([(root,1)] if root else []); step=0
    while q:
        before=[(node.val,depth) for node,depth in q]
        node,depth=q.popleft(); step+=1
        leaf=node.left is None and node.right is None
        if not leaf:
            if node.left:q.append((node.left,depth+1))
            if node.right:q.append((node.right,depth+1))
        after=[(node.val,d) for node,d in q]
        print(step,'queue trước',before,'xét',(node.val,depth),'lá?',leaf,'sau',after)
        if leaf:return depth
    return 0
assert bfs_queue_snapshot(tree([1,2,3,4,None,None,5]))==3

### 7.2. Một trace khác: lá nông và nhánh cực sâu

Không nên chỉ trace cây cân bằng: nó không cho thấy lợi thế dừng sớm.
Tạo cây gốc 10, con trái là lá 5, con phải nối 20→30→40→50.
BFS phải lấy gốc 10 (tầng 1), rồi nút 5 (tầng 2) và kết thúc, dù nhánh bên phải vẫn chưa xử lý.
Nếu xếp con phải vào queue trước con trái, ta vẫn chỉ thấy lá ở tầng 2 sau vài bước;
đáp án **không đổi**, vì queue FIFO luôn ưu tiên tầng 2 trước tầng 3.

Đây là input thiết kế để phân biệt thuật toán đúng với cách duyệt sâu ngẫu nhiên.

In [ ]:
long_right=TreeNode(10,TreeNode(5),TreeNode(20))
cur=long_right.right
for v in (30,40,50):
    cur.right=TreeNode(v)
    cur=cur.right
print('BFS trả:',solve(long_right))
print('DFS baseline trả:',baseline(long_right))
assert solve(long_right)==baseline(long_right)==2

In [ ]:
def trace_min(root):
    if root is None:return 0
    q=deque([(root,1)])
    while q:
        node,depth=q.popleft()
        print('pop:',node.val,'độ sâu',depth,'queue còn:',[(x.val,d) for x,d in q])
        if node.left is None and node.right is None:
            print('ĐÂY LÀ LÁ ĐẦU TIÊN ->',depth);return depth
        if node.left:q.append((node.left,depth+1))
        if node.right:q.append((node.right,depth+1))
trace_min(tree([3,9,20,None,None,15,7]))

### 7.3. BFS in cả quyết định mở rộng trái/phải

Trong `trace_min` ta biết mỗi `pop` và phần queue còn lại.
Ta có thể tăng tính quan sát bằng cách in cả **những nút con được thêm**:
trước khi thêm ghi rõ độ sâu cũ, sau khi thêm ghi danh sách queue mới.
Kỹ thuật trace này áp dụng cho các bài đường đi ngắn nhất không trọng số,
đặc biệt khi bạn cần tìm xem mình đã *bỏ quên con* hay *thêm con trống* tại bước nào.

Code ở đây không phải lời giải submit; nó là máy ghi nhật ký các trạng thái.

In [ ]:
def trace_bfs_children(root):
    if root is None:return 0
    q=deque([(root,1)])
    while q:
        node,depth=q.popleft()
        print('xét',node.val,'ở tầng',depth)
        if node.left is None and node.right is None:
            print('đã đến lá ->',depth);return depth
        for label,child in (('trái',node.left),('phải',node.right)):
            if child is not None:
                q.append((child,depth+1))
                print('   thêm con',label,child.val,'vào cuối queue')
    return 0
assert trace_bfs_children(tree([1,2,3,None,4]))==2

### 7.4. So BFS với DFS lặp sử dụng stack

BFS và DFS đều có thể tìm độ sâu nhỏ nhất; **sự khác nhau** là thứ tự duyệt và thời điểm biết đáp án.
DFS lặp dùng `stack.pop()` (LIFO), có thể gặp lá sâu trước, nên phải giữ `best` và duyệt tiếp;
không thể return tại lá đầu tiên. BFS dùng `queue.popleft()` (FIFO), có thể return tại lá đầu tiên.

Khi DFS đã biết một đáp án `best`, có thể tỉa nhánh ở độ sâu ≥ best;
nhưng ở bài nhập môn này ta ưu tiên code dễ kiểm chứng, không cần tối ưu hóa đó.
Dùng hàm dưới để thấy hai cách đều trả cùng kết quả trên cây nhỏ.

In [ ]:
def min_depth_dfs_stack(root):
    if root is None:return 0
    stack=[(root,1)]; best=float('inf')
    while stack:
        node,depth=stack.pop()
        if node.left is None and node.right is None:
            best=min(best,depth)
            continue
        if node.left:stack.append((node.left,depth+1))
        if node.right:stack.append((node.right,depth+1))
    return best
for values in ([],[1],[1,2,3],[1,None,2,None,3]):
    assert min_depth_dfs_stack(tree(values)) == solve(tree(values))
print('BFS và DFS stack cho cùng kết quả ở 4 trường hợp')

## 8. Vì sao thuật toán đúng? — lập luận bằng bất biến

Queue FIFO đảm bảo các node được xét theo từng tầng từ thấp đến cao. Trước khi xét node độ sâu `d`, mọi node độ sâu nhỏ hơn `d` đã được xét và không phải lá (nếu có lá đã return). Vì vậy lá đầu tiên có độ sâu nhỏ nhất.

**Hai câu cần trả lời:** (1) Sau mỗi bước, điều gì vẫn đúng? (2) Khi dừng, vì sao giá trị trả về chính là đáp án theo yêu cầu?

## 9. Code nộp LeetCode (Python 3)

**Chỉ lấy cell bên dưới để nộp.** Phần dựng input và `print()` ở các cell khác chỉ phục vụ học tập.

In [ ]:
from typing import Optional
from collections import deque
class Solution:
    def minDepth(self, root: Optional[TreeNode]):
        if root is None:return 0
        q=deque([(root,1)])
        while q:
            node,depth=q.popleft()
            if node.left is None and node.right is None:
                return depth
            if node.left is not None:q.append((node.left,depth+1))
            if node.right is not None:q.append((node.right,depth+1))
        return 0

## 10. Kiểm thử đúng đầu ra, trường hợp biên và lỗi dễ bỏ sót

Các lệnh `assert` sẽ dừng với thông báo lỗi nếu kết quả **khác** giá trị mong đợi. Tập test bao gồm ví dụ dễ hiểu, input nhỏ nhất và tình huống dễ nhầm. Đừng chỉ kiểm tra một ví dụ.

In [ ]:
for a,expect in [([],0),([1],1),([3,9,20,None,None,15,7],2),([2,None,3,None,4],3),([1,2],2)]:
    assert solve(tree(a))==expect
    assert Solution().minDepth(tree(a))==expect
print('PASS 5 ca minDepth')

### 10.1. Tạo ma trận kiểm thử có chủ ý cho minDepth

**Test không chỉ là ví dụ thông thường.** Ta cần nhóm các trường hợp để buộc thuật toán đi qua nhánh khó:
(1) Cây rỗng; (2) Một nút; (3) Chỉ một nhánh trái; (4) Chỉ một nhánh phải;
(5) Cây đầy; (6) Lá rất nông nằm bên cạnh nhánh sâu.

Mỗi test cần `expected` tính từ cấu trúc cây chứ không tính theo suy đoán.
So sánh `baseline`, BFS `solve` và bản LeetCode `Solution().minDepth`.
Với test cây một bên rỗng, `wrong_min_depth_demo` ở trên phải cho kết quả khác,
đó là dấu hiệu test thực sự có khả năng bắt lỗi.

In [ ]:
min_cases=[
    ([],0),([5],1),([1,2],2),([1,None,2],2),
    ([1,2,3],2),([1,None,2,None,3,None,4],4),
    ([1,2,3,None,None,4,None,5],2)
]
for values,expected in min_cases:
    candidates=[baseline(tree(values)),solve(tree(values)),Solution().minDepth(tree(values))]
    assert candidates==[expected]*3,(values,candidates,expected)
print('PASS',len(min_cases),'test có chủ đích')

### 10.2. Kiểm thử ngẫu nhiên có đối chiếu (randomized differential test)

Để kiểm tra những hình dạng khó tự nghĩ ra, ta tạo các cây nhỏ ngẫu nhiên,
đối chiếu BFS với DFS baseline (hai tư duy khác nhau).
**Seed cố định** khiến kết quả tái lập được khi người khác mở notebook.

Kiểm thử ngẫu nhiên không thay thế chứng minh tính đúng: nó chỉ giúp tìm phản ví dụ.
Nếu phát hiện sai khác, hãy in cây nhỏ nhất gặp được và trace queue/stack tại đó.
Bài luyện thêm: thử đổi xác suất tạo nhánh để có nhiều cây lệch hơn.

In [ ]:
import random
rng111=random.Random(111)
def random_tree111(depth, stop=0.25):
    if depth==0 or rng111.random()<stop:return None
    return TreeNode(rng111.randint(-8,8),random_tree111(depth-1,stop),random_tree111(depth-1,stop))
for _ in range(180):
    candidate=random_tree111(6,0.28)
    assert solve(candidate)==baseline(candidate)==min_depth_dfs_stack(candidate)
print('PASS 180 cây ngẫu nhiên')

## 11. Phân tích Time/Space Complexity

`n` số node. **Time O(n)** tệ nhất (BFS có thể dừng sớm); **Space O(w)** với `w` là số node lớn nhất của tầng đang chờ, tối đa O(n). Baseline DFS O(n) time, O(h) stack.

Nhớ phân biệt **số phần tử đầu vào** với **độ cao cây**, **số bit** hoặc **số chữ số**. Nếu thuật toán dùng recursion, bộ nhớ ngăn xếp lời gọi cũng cần được tính.

## 12. Các lỗi người mới hay mắc

- ❌ `min(depth(left),depth(right))+1` khi một nhánh None: trả 1 cho cây lệch.
- ❌ Dừng ở node có ít nhất một con None: chưa chắc là lá.
- ❌ `list.pop(0)` thay `deque.popleft()` liên tục gây tốn chi phí.

Khi debug: cố tạo input nhỏ nhất gây sai, trace biến ở **ngay trước lệnh `if`/`return`**, rồi sửa nguyên nhân thay vì thêm điều kiện tùy tiện.

### 12.1. Mổ xẻ code sai: nhầm một nhánh vắng với lá

Ví dụ sai: `if node.left is None or node.right is None: return depth`.
`or` trả True khi chỉ một vế đúng; node chỉ thiếu con trái vẫn có thể nối xuống con phải,
nên **chưa phải lá**. Điều kiện đúng cần **`and`**: cả hai con cùng `None`.

Dạy một cú pháp mới phải gắn với phản ví dụ nhỏ. Ta dùng cây `2 → 3 → 4`
và so hai điều kiện trên nút `2`; nếu thay `and` bằng `or` thì chương trình dừng ngay ở gốc.
Lỗi rất dễ vượt qua test cây đầy, nên cần test cây lệch.

In [ ]:
node_bug=tree([2,None,3,None,4])
print('thiếu ít nhất một con? (sai khi coi là lá):',node_bug.left is None or node_bug.right is None)
print('thiếu cả hai con? (đúng định nghĩa lá):',node_bug.left is None and node_bug.right is None)
assert (node_bug.left is None or node_bug.right is None)
assert not (node_bug.left is None and node_bug.right is None)

### 12.2. `list.pop(0)` so với `deque.popleft()` và vấn đề thời gian

Cả `list.pop(0)` và `deque.popleft()` đều trả phần tử đầu,
nhưng `list.pop(0)` thường phải dời những phần tử còn lại, có chi phí O(k) nếu list dài k.
Còn `deque.popleft()` lấy đầu hàng đợi trong O(1).
Đổi cấu trúc dữ liệu **không đổi đáp án** nhưng có thể đổi độ phức tạp thời gian.

Đối với một cây có tầng rất rộng, lặp `pop(0)` có thể khiến cách duyệt trở nên kém hiệu quả;
đừng tự động kết luận `BFS=O(n)` nếu cách triển khai queue không đạt O(1) cho lấy đầu.
Lưu ý: với một list nhỏ, cả hai vẫn cho cùng thứ tự lấy ra.

In [ ]:
list_queue=[1,2,3]
deque_queue=deque([1,2,3])
while list_queue:
    x=list_queue.pop(0)
    y=deque_queue.popleft()
    print('hai queue trả:',x,y)
    assert x==y
assert not deque_queue

## 13. Nhận diện dạng bài và vận dụng

Đề hỏi **ngắn nhất từ gốc đến sự kiện đầu tiên** trong cấu trúc không trọng số → BFS rất hợp, vì có thể return ngay tại node đích.

**Chuyển giao:** hãy tìm một bài khác có cùng dấu hiệu đầu vào và tập trung vào lý do giữ bất biến, không học thuộc tên biến.

## 14. Biến thể có ý nghĩa

Tạo cây lệch một phía và cây hai nhánh, so sánh **minDepth** với **maxDepth**: khi nào chúng bằng nhau?

Biến thể này dùng lại kỹ thuật vừa học, nhưng có một thay đổi nhỏ giúp kiểm tra bạn đã hiểu hay chưa.

In [ ]:
for a in [[1],[2,None,3,None,4],[1,2,3]]:
    r=tree(a);print(a,'min=',solve(r))

### 14.1. Biến thể: trả đường đi ngắn nhất, không chỉ độ dài

Bài #111 chỉ yêu cầu con số, nên lưu `(node,depth)` là đủ.
Nếu đề yêu cầu **một đường gốc→lá ngắn nhất**, ta có thể lưu thêm đường đi đến mỗi node.
BFS vẫn đảm bảo đường tìm thấy đầu tiên là ngắn nhất theo số nút.
Với minh họa nhỏ, tuple `(node, path)` dễ hiểu hơn việc lưu `parent` map.
Đổi lại, mỗi lần nối list đường đi tạo bản sao, tốn bộ nhớ hơn — đây là **biến thể học tập**, không phải code submit tối ưu.

Ví dụ `[1,2,3,None,None,4]` phải trả `[1,2]`, không cần liệt kê mọi đường dài hơn.

In [ ]:
def shortest_root_leaf_path(root):
    if root is None:return []
    q=deque([(root,[root.val])])
    while q:
        node,path=q.popleft()
        if node.left is None and node.right is None:return path
        if node.left:q.append((node.left,path+[node.left.val]))
        if node.right:q.append((node.right,path+[node.right.val]))
    return []
assert shortest_root_leaf_path(tree([1,2,3,None,None,4]))==[1,2]
print('đường đi ngắn nhất:',shortest_root_leaf_path(tree([1,2,3,None,None,4])))

### 14.2. Khi nào bài yêu cầu maxDepth thay vì minDepth?

**minDepth** cần tìm lá ở độ sâu nhỏ nhất; BFS có thể dừng ở lá đầu.
**maxDepth** cần độ sâu xa nhất, nên không thể dừng ở lá đầu; phải xét đủ những nhánh có thể sâu hơn.
Một cây một nút có minDepth = maxDepth = 1. Cây hình dây cũng có hai kết quả bằng nhau.
Cây có một lá nông bên trái và lá sâu bên phải cho kết quả khác nhau.

Khi chuyển pattern, đừng máy móc thay `min` bằng `max` trong code chứa điều kiện dừng sớm.
Tư duy đúng là: **đích của bài là lá đầu tiên hay lá xa nhất?**

In [ ]:
def max_depth111(node):
    if node is None:return 0
    return 1+max(max_depth111(node.left),max_depth111(node.right))
for values in ([1],[1,None,2,None,3],[1,2,3,None,None,4,None,5]):
    root=tree(values)
    print(values,'min=',solve(root),'max=',max_depth111(root))
    assert solve(root)<=max_depth111(root)

## 15. Tự luyện — 5 bài tăng dần

**Mục tiêu:** tự hiểu cú pháp, trace bằng tay, viết lại thuật toán, chuyển sang biến thể, rồi thực hiện mini challenge. Tất cả ví dụ giữ nguyên đúng bản chất đề LeetCode. **Đừng xem đáp án trước.**

### Bài 1 — Dự đoán output của Python (dễ)

Với `r=tree([2,None,3])` hãy đoán `r.left is None`, `r.right is None`, và `r.left is None and r.right is None`.

Hãy viết output ra giấy trước khi chạy cell dưới.

In [ ]:
r=tree([2,None,3])
print(r.left is None)
print(r.right is None)
print(r.left is None and r.right is None)

### Bài 2 — Trace bằng tay (hiểu bản chất)

Trace `[2,None,3,None,4]`: nếu lấy `min(độ sâu trái, độ sâu phải)` tại gốc sẽ sai thế nào? Độ sâu tối thiểu thật là mấy?

Tự viết bảng gồm **trạng thái trước**, phép kiểm tra và **trạng thái sau**; không chỉ ghi kết quả cuối.

### Bài 3 — TODO: Tự viết lời giải chính

Viết `exercise_min_depth(root)` bằng đệ quy, xử lý riêng trường hợp node chỉ có **một** con.

Các hàm bài tập được định nghĩa độc lập và **không được gọi** khi chưa hoàn thiện.

In [ ]:
def exercise_min_depth(root):
    # TODO: root None, chỉ có trái, chỉ có phải, có cả hai.
    raise NotImplementedError

### Bài 4 — TODO: Chuyển giao sang biến thể

Giải bằng BFS (`deque`) trong `exercise_min_depth_bfs(root)`: lá **đầu tiên** lấy ra là lá có độ sâu nhỏ nhất.

Hãy thử input nhỏ nhất và một phản ví dụ. Dự đoán output trước khi gọi.

In [ ]:
def exercise_min_depth_bfs(root):
    # TODO: queue chứa (node,depth); khi gặp lá thì trả depth ngay.
    raise NotImplementedError

### Bài 5 — TODO: Mini challenge

Mini challenge: `exercise_nearest_leaf(root)` trả `(độ_sâu, giá_trị_lá)` cho lá gần gốc nhất; nếu cùng độ sâu, ưu tiên lá bên trái; cây rỗng trả `(0,None)`.

Hãy nêu độ phức tạp của hàm và thử tối thiểu 3 ca biên.

In [ ]:
def exercise_nearest_leaf(root):
    # TODO: dùng BFS; đưa con trái vào queue trước con phải.
    raise NotImplementedError

### Bài 6 — TODO: đếm bao nhiêu lá có độ sâu nhỏ nhất

Bài mới để **chuyển giao** kỹ thuật BFS, không chỉ thay đổi tên biến:
viết `exercise_count_nearest_leaves(root) -> (min_depth, number_of_leaves)`.
Nếu cây rỗng, trả `(0,0)`. Với cây `[1,2,3]`, hai lá đều ở tầng 2 nên trả `(2,2)`.
Với `[1,2,3,None,None,4]`, lá nông nhất chỉ là nút 2 nên trả `(2,1)`.

**Gợi ý cấp 1:** BFS được duyệt theo tầng, đừng dừng ngay khi thấy lá đầu vì có thể có thêm lá cùng tầng.
**Gợi ý cấp 2:** lưu `min_d` lần đầu gặp lá; đếm tiếp các lá có depth bằng `min_d`.
**Gợi ý cấp 3:** khi queue bắt đầu chứa depth lớn hơn `min_d`, có thể kết thúc vì không còn lá cùng tầng.

In [ ]:
def exercise_count_nearest_leaves(root):
    # TODO: BFS và đếm số lá ở tầng nông nhất.
    # Không gọi solve() ở đây; tự giải theo trạng thái queue.
    raise NotImplementedError

# Tự làm xong mới bỏ comment hai test sau:
# assert exercise_count_nearest_leaves(None)==(0,0)
# assert exercise_count_nearest_leaves(tree([1,2,3]))==(2,2)

## 16. Gợi ý mở dần — đừng đọc cùng lúc

<details><summary><b>Cấp 1 · Trực giác</b></summary>

**Lá** là node có cả hai con rỗng; con rỗng không phải đường đi có thể kết thúc.

</details>

<details><summary><b>Cấp 2 · Dữ liệu và cú pháp</b></summary>

BFS dùng `deque`; muốn ưu tiên lá trái, `append(left)` trước `append(right)`.

</details>

<details><summary><b>Cấp 3 · Gần-code</b></summary>

Đệ quy: nếu thiếu trái, chỉ có thể đi phải; nếu thiếu phải, chỉ có thể đi trái; nếu có đủ hai mới lấy `min`.

</details>

## 17. Đáp án tham khảo — chỉ đọc sau khi tự làm

### Bài 1 — Output

Ba dòng: `True`, `False`, `False`. Node `2` **không phải lá** dù thiếu nhánh trái.

### Bài 2 — Trace tay

Nhánh trái trống có độ sâu 0 **không tạo đường đi tới lá**. Nhánh phải nối `2→3→4` có 3 node, nên đáp án **3**, không phải `1+min(0,2)=1`.

### Đáp án bài 3 — Viết lại giải pháp chính

Đọc rõ base case và ý nghĩa giá trị trả về. Đáp án độc lập, không dựa vào `solve` phía trên.

In [ ]:
def answer_min_depth(root):
    if root is None: return 0
    if root.left is None: return 1+answer_min_depth(root.right)
    if root.right is None: return 1+answer_min_depth(root.left)
    return 1+min(answer_min_depth(root.left),answer_min_depth(root.right))

### Đáp án bài 4 — Biến thể

So sánh **trạng thái cần mang theo** giữa bài chính và biến thể.

In [ ]:
def answer_min_depth_bfs(root):
    if root is None: return 0
    q=deque([(root,1)])
    while q:
        node,depth=q.popleft()
        if node.left is None and node.right is None: return depth
        if node.left: q.append((node.left,depth+1))
        if node.right: q.append((node.right,depth+1))

### Đáp án bài 5 — Mini challenge

Kiểm tra kỹ trường hợp rỗng, đầu/cuối, và các điều kiện dừng.

In [ ]:
def answer_nearest_leaf(root):
    if root is None: return (0,None)
    q=deque([(root,1)])
    while q:
        node,depth=q.popleft()
        if node.left is None and node.right is None: return depth,node.val
        if node.left: q.append((node.left,depth+1))
        if node.right: q.append((node.right,depth+1))

### Đáp án bài 6 — BFS một tầng có thể chứa nhiều lá

Khi gặp lá đầu tiên ở depth d, ta đã biết đáp án minDepth là d,
nhưng chưa biết có bao nhiêu lá khác ở **cùng tầng d** đang nằm trong queue.
Ta tiếp tục duyệt cho tới khi gặp phần tử đầu tiên ở tầng lớn hơn d, rồi trả `(d,count)`.

Điểm chính so với bài #111: yêu cầu mới **đếm tất cả lá tối ưu**, không chỉ tìm một.
Độ phức tạp thời gian tệ nhất O(n); bộ nhớ queue O(w), với w là số nút của tầng đông nhất.

In [ ]:
def answer_count_nearest_leaves(root):
    if root is None:return (0,0)
    q=deque([(root,1)])
    min_d=None;count=0
    while q:
        node,depth=q.popleft()
        if min_d is not None and depth>min_d:
            break
        if node.left is None and node.right is None:
            min_d=depth
            count+=1
        else:
            if node.left:q.append((node.left,depth+1))
            if node.right:q.append((node.right,depth+1))
    return min_d,count
for arr,expected in [([],(0,0)),([1],(1,1)),([1,2,3],(2,2)),
                     ([1,2,3,None,None,4],(2,1))]:
    assert answer_count_nearest_leaves(tree(arr))==expected
print('PASS bài 6: đếm số lá nông nhất')

### Kiểm thử 3 lời giải bài tập

Các assert sau **chỉ kiểm tra đáp án mẫu**, không chạy các hàm TODO của bạn. Khi hoàn thiện TODO, bạn hãy tự thay tên hàm trong test để đánh giá.

In [ ]:
for arr,expect in [([],0),([1],1),([2,None,3,None,4],3),([3,9,20,None,None,15,7],2)]:
    assert answer_min_depth(tree(arr))==expect
    assert answer_min_depth_bfs(tree(arr))==expect
assert answer_nearest_leaf(tree([3,9,20,None,None,15,7]))==(2,9)
assert answer_nearest_leaf(None)==(0,None)
print("PASS: đáp án 3 bài coding của phần luyện tập")

## 18. Tóm tắt và checklist submit

**Nhớ nhanh:** BFS duyệt theo **thứ tự độ sâu tăng dần**. Lá đầu tiên lấy ra khỏi queue chính là lá nông nhất: có thể dừng sớm ngay khi gặp.

- [ ] Tôi tự nói được input, output và điều kiện biên.
- [ ] Tôi hiểu từng cú pháp Python trong thuật toán.
- [ ] Tôi có thể tự vẽ bảng trace ít nhất một ví dụ.
- [ ] Tôi giải thích được **tại sao đúng** mà không dựa vào test.
- [ ] Tôi phân biệt được Time/Space của baseline và lời giải chính.
- [ ] Tôi copy đúng phần class LeetCode, không copy hàm dựng cây/list hay `print()`.
- [ ] Tôi tự làm xong các TODO, rồi mới đọc đáp án.

**Bài tiếp theo:** duy trì thói quen trace từ trạng thái nhỏ và chỉ tối ưu khi đã hiểu bản chất.